# NASA HLS and OPERA Storm-Debris Mapping

Use HLS optical imagery, OPERA RTC-S1 radar imagery, NLCD forest cover, and optional Local Moran's I to map likely post-storm debris concentrations.

Run the notebook from top to bottom. Each prompt shows a default value that can be accepted by pressing **Enter**.


## 1. Import packages

Load the libraries used by the workflow.


In [ ]:
# Import the packages used throughout the workflow.
import base64
import contextlib
import json
import math
import sys
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass, field
from datetime import UTC, datetime
from pathlib import Path
from urllib.parse import urlparse

import earthaccess
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import shapely
import yaml
from ipyleaflet import DrawControl, GeoJSON, ImageOverlay, LayersControl, Map, basemaps
from IPython.display import Image, display
from rasterio.crs import CRS as RasterioCRS
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import Affine, from_origin
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from shapely.geometry import box, shape


## 2. Enter the analysis settings

Define the study area, dates, imagery filters, calculations, performance settings, and optional spatial analysis.


In [ ]:
# Define small helpers for checked user input.
def ask(question, default=None, parser=str, validate=None):
    """Prompt until the user enters a valid value."""
    suffix = "" if default is None else f" [{default}]"
    while True:
        raw = input(f"{question}{suffix}: ").strip()
        if not raw:
            if default is None:
                print("  A value is required.")
                continue
            raw = str(default)
        try:
            value = parser(raw)
        except (TypeError, ValueError) as error:
            print(f"  Could not read that value: {error}")
            continue
        if validate is not None and not validate(value):
            print("  That value is not valid for this setting.")
            continue
        return value


def parse_bool(text):
    """Read common yes/no values."""
    value = text.strip().lower()
    if value in {"y", "yes", "true", "1"}:
        return True
    if value in {"n", "no", "false", "0"}:
        return False
    raise ValueError("enter yes or no")


def parse_list(text, item_parser=str):
    """Read a comma-separated list."""
    return [item_parser(item.strip()) for item in text.split(",") if item.strip()]


def parse_mapping(text, value_parser=float):
    """Read comma-separated name=value pairs."""
    output = {}
    for item in text.split(","):
        if not item.strip():
            continue
        if "=" not in item:
            raise ValueError(f"missing '=' in {item!r}")
        name, value = item.split("=", 1)
        output[name.strip().lower()] = value_parser(value.strip())
    return output


def ask_choice(question, choices, default):
    """Prompt for one value from a fixed set."""
    allowed = {choice.lower(): choice.lower() for choice in choices}
    return ask(
        f"{question} ({'/'.join(choices)})",
        default=default,
        parser=lambda text: text.strip().lower(),
        validate=lambda value: value in allowed,
    )


SUPPORTED_OPTICAL_INDICES = ("ndvi", "evi", "nbr", "ndmi", "dfi", "ndti")
SUPPORTED_RADAR_INDICES = ("vv_db", "vh_db", "rvi", "mrdfi", "vh_vv_ratio", "dprvi")
SUPPORTED_MASK_FLAGS = (
    "cirrus", "cloud", "adjacent_cloud", "cloud_shadow", "snow", "high_aerosol"
)


In [ ]:
# Ask for the study area, dates, grid, and forest layer.
output_directory = ask(
    "Output folder",
    default="debris_output",
    validate=lambda value: bool(value.strip()),
)

baseline_years = ask(
    "Baseline years, separated by commas",
    default="2021,2022,2023",
    parser=lambda text: parse_list(text, int),
    validate=lambda years: len(years) >= 2 and all(1980 <= year <= 2100 for year in years),
)
target_years = ask(
    "Target years, separated by commas",
    default="2024,2025",
    parser=lambda text: parse_list(text, int),
    validate=lambda years: (
        bool(years)
        and all(1980 <= year <= 2100 for year in years)
        and not set(years).intersection(baseline_years)
    ),
)
season_start = ask(
    "Season start as MM-DD",
    default="10-01",
    validate=lambda value: len(value) == 5 and value[2] == "-",
)
season_end = ask(
    "Season end as MM-DD",
    default="11-30",
    validate=lambda value: len(value) == 5 and value[2] == "-",
)

analysis_crs = ask("Analysis CRS", default="EPSG:5070")
analysis_resolution_m = ask(
    "Analysis pixel size in meters",
    default=30,
    parser=float,
    validate=lambda value: value > 0,
)

aoi_method = ask_choice(
    "Study-area method",
    choices=("fips", "vector", "drawn"),
    default="fips",
)
county_fips = []
aoi_path = ""
tiger_year = 2023

if aoi_method == "fips":
    county_fips = ask(
        "County FIPS codes, separated by commas",
        default=(
            "37005,37009,37011,37021,37023,37027,37039,37043,37075,37087,"
            "37089,37099,37111,37113,37115,37121,37149,37161,37171,37173,"
            "37175,37189,37193,37197,37199"
        ),
        parser=lambda text: [item.zfill(5) for item in parse_list(text)],
        validate=lambda values: bool(values) and all(len(value) == 5 for value in values),
    )
    tiger_year = ask(
        "TIGER county-boundary year",
        default=2023,
        parser=int,
        validate=lambda value: 2010 <= value <= 2100,
    )
elif aoi_method == "vector":
    aoi_path = ask(
        "Path to the AOI vector file",
        validate=lambda value: Path(value).is_file(),
    )

nlcd_source_path = ask(
    "Path to the local NLCD raster",
    validate=lambda value: Path(value).is_file(),
)
nlcd_product_year = ask(
    "NLCD product year",
    default=2024,
    parser=int,
    validate=lambda value: 2001 <= value <= 2100,
)
forest_classes = ask(
    "NLCD forest classes, separated by commas",
    default="41,42,43",
    parser=lambda text: parse_list(text, int),
    validate=lambda values: bool(values),
)
nlcd_resampling = ask_choice(
    "NLCD resampling",
    choices=("nearest", "mode"),
    default="nearest",
)


In [ ]:
# Ask for imagery, masking, index, baseline, and composite settings.
hls_products = ask(
    "HLS products, separated by commas",
    default="HLSL30,HLSS30",
    parser=lambda text: [item.upper() for item in parse_list(text)],
    validate=lambda values: bool(values) and set(values).issubset({"HLSL30", "HLSS30"}),
)
max_cloud_cover = ask(
    "Maximum HLS granule cloud cover percent",
    default=20,
    parser=float,
    validate=lambda value: 0 <= value <= 100,
)
hls_minimum_observations = ask(
    "Minimum valid HLS observations per annual composite pixel",
    default=3,
    parser=int,
    validate=lambda value: value >= 1,
)
hls_indices = ask(
    "Optical indices, separated by commas",
    default=",".join(SUPPORTED_OPTICAL_INDICES),
    parser=lambda text: [item.lower() for item in parse_list(text)],
    validate=lambda values: bool(values) and set(values).issubset(SUPPORTED_OPTICAL_INDICES),
)
mask_flag_names = ask(
    "HLS quality flags to mask, separated by commas",
    default=",".join(SUPPORTED_MASK_FLAGS),
    parser=lambda text: [item.lower() for item in parse_list(text)],
    validate=lambda values: set(values).issubset(SUPPORTED_MASK_FLAGS),
)
reflectance_scale = ask(
    "HLS reflectance scale",
    default=0.0001,
    parser=float,
    validate=lambda value: value > 0,
)

opera_short_name = ask(
    "OPERA RTC-S1 collection short name",
    default="OPERA_L2_RTC-S1_V1",
)
opera_required_assets = ask(
    "Required OPERA assets, separated by commas",
    default="VV,VH,mask",
    parser=lambda text: ["mask" if item.lower() == "mask" else item.upper() for item in parse_list(text)],
    validate=lambda values: set(values) == {"VV", "VH", "mask"},
)
opera_minimum_observations = ask(
    "Minimum valid OPERA observations per annual composite pixel",
    default=3,
    parser=int,
    validate=lambda value: value >= 1,
)
radar_indices = ask(
    "Radar indices, separated by commas",
    default=",".join(SUPPORTED_RADAR_INDICES),
    parser=lambda text: [item.lower() for item in parse_list(text)],
    validate=lambda values: bool(values) and set(values).issubset(SUPPORTED_RADAR_INDICES),
)
opera_valid_mask_value = ask(
    "OPERA validity-mask value to keep",
    default=0,
    parser=int,
)
opera_resampling = ask_choice(
    "OPERA continuous-data resampling",
    choices=("nearest", "bilinear", "cubic"),
    default="bilinear",
)

minimum_baseline_year_count = ask(
    "Minimum baseline years required",
    default=min(3, len(baseline_years)),
    parser=int,
    validate=lambda value: 2 <= value <= len(baseline_years),
)
mad_scale = ask(
    "MAD scale factor",
    default=1.4826,
    parser=float,
    validate=lambda value: value > 0,
)

spread_floor_default = (
    "ndvi=0.02,evi=0.02,nbr=0.02,ndmi=0.02,dfi=1.0,ndti=0.02,"
    "vv_db=0.5,vh_db=0.5,rvi=0.02,mrdfi=0.02,vh_vv_ratio=0.02,dprvi=0.02"
)
minimum_spread_floors = ask(
    "Minimum robust-spread floors as name=value pairs",
    default=spread_floor_default,
    parser=parse_mapping,
)

percentage_excluded_variables = ask(
    "Variables excluded from percentage change",
    default="dfi,vv_db,vh_db",
    parser=lambda text: [item.lower() for item in parse_list(text)],
)
percentage_floor_default = (
    "ndvi=0.05,evi=0.05,nbr=0.05,ndmi=0.05,dfi=1.0,ndti=0.05,"
    "vv_db=1.0,vh_db=1.0,rvi=0.05,mrdfi=0.05,vh_vv_ratio=0.05,dprvi=0.05"
)
percentage_minimum_baseline_magnitude = ask(
    "Percentage-change denominator floors as name=value pairs",
    default=percentage_floor_default,
    parser=parse_mapping,
)

spectral_greenness = ask(
    "Greenness-group indices",
    default="ndvi,evi",
    parser=lambda text: [item.lower() for item in parse_list(text)],
    validate=lambda values: bool(values),
)
spectral_moisture = ask(
    "Moisture-group indices",
    default="nbr,ndmi",
    parser=lambda text: [item.lower() for item in parse_list(text)],
    validate=lambda values: bool(values),
)
spectral_exposed = ask(
    "Exposed-material-group indices",
    default="dfi,ndti",
    parser=lambda text: [item.lower() for item in parse_list(text)],
    validate=lambda values: bool(values),
)
radar_inputs = ask(
    "Radar indices used in the radar score",
    default="rvi,mrdfi,dprvi",
    parser=lambda text: [item.lower() for item in parse_list(text)],
    validate=lambda values: bool(values) and set(values).issubset(radar_indices),
)
spectral_disturbance_direction = ask(
    "Optical disturbance directions as name=1 or name=-1",
    default="ndvi=-1,evi=-1,nbr=-1,ndmi=-1,dfi=1,ndti=1",
    parser=lambda text: parse_mapping(text, int),
    validate=lambda values: all(value in {-1, 1} for value in values.values()),
)
z_cap = ask(
    "Maximum robust-anomaly magnitude used in 0-1 scores",
    default=3.0,
    parser=float,
    validate=lambda value: value > 0,
)


In [ ]:
# Ask for processing, display, and optional Local Moran settings.
analysis_chunk_pixels = ask(
    "Analysis block size in pixels",
    default=1536,
    parser=int,
    validate=lambda value: value >= 128,
)
composite_block_pixels = ask(
    "Composite block size in pixels",
    default=384,
    parser=int,
    validate=lambda value: value >= 64,
)
preview_max_dimension_pixels = ask(
    "Maximum preview-map dimension in pixels",
    default=1000,
    parser=int,
    validate=lambda value: value >= 200,
)
distribution_sample_size = ask(
    "Maximum sampled pixels per distribution plot",
    default=20000,
    parser=int,
    validate=lambda value: value >= 100,
)
distribution_seed = ask(
    "Distribution-sampling random seed",
    default=0,
    parser=int,
)

run_local_moran = ask(
    "Run Local Moran's I",
    default="yes",
    parser=parse_bool,
)
lisa_variable = "index_composite_combined"
lisa_support_resolution_m = 90.0
lisa_permutations = 99
lisa_alpha = 0.05
lisa_weights_method = "queen"
lisa_row_standardize = True
lisa_minimum_valid_fraction = 0.5
lisa_maximum_units = 250000
lisa_random_seed = 42
lisa_alternative = "two-sided"
lisa_n_jobs = 1

if run_local_moran:
    lisa_variable = ask_choice(
        "Raster used for Local Moran's I",
        choices=(
            "index_composite_combined",
            "index_composite_spectral",
            "index_composite_radar",
        ),
        default="index_composite_combined",
    )
    lisa_support_resolution_m = ask(
        "Local Moran support resolution in meters",
        default=90,
        parser=float,
        validate=lambda value: value >= analysis_resolution_m,
    )
    lisa_permutations = ask(
        "Local Moran permutations",
        default=99,
        parser=int,
        validate=lambda value: value >= 0,
    )
    lisa_alpha = ask(
        "Local Moran significance level",
        default=0.05,
        parser=float,
        validate=lambda value: 0 < value < 1,
    )
    lisa_weights_method = ask_choice(
        "Local Moran neighbors",
        choices=("queen", "rook"),
        default="queen",
    )
    lisa_row_standardize = ask(
        "Row-standardize Local Moran weights",
        default="yes",
        parser=parse_bool,
    )
    lisa_minimum_valid_fraction = ask(
        "Minimum valid fraction in each Local Moran support cell",
        default=0.5,
        parser=float,
        validate=lambda value: 0 < value <= 1,
    )
    lisa_maximum_units = ask(
        "Maximum Local Moran analysis units",
        default=250000,
        parser=int,
        validate=lambda value: value >= 1000,
    )
    lisa_random_seed = ask("Local Moran random seed", default=42, parser=int)
    lisa_alternative = ask_choice(
        "Local Moran alternative",
        choices=("two-sided", "greater", "less"),
        default="two-sided",
    )
    lisa_n_jobs = ask(
        "Local Moran worker processes",
        default=1,
        parser=int,
        validate=lambda value: value >= 1,
    )

# Confirm that every selected variable has the settings needed later.
selected_variables = set(hls_indices) | set(radar_indices)
missing_spread_floors = selected_variables.difference(minimum_spread_floors)
if missing_spread_floors:
    raise ValueError(f"Missing spread floors for: {sorted(missing_spread_floors)}")

percentage_variables = selected_variables.difference(percentage_excluded_variables)
missing_percentage_floors = percentage_variables.difference(
    percentage_minimum_baseline_magnitude
)
if missing_percentage_floors:
    raise ValueError(f"Missing percentage floors for: {sorted(missing_percentage_floors)}")

spectral_groups = {
    "greenness": spectral_greenness,
    "moisture": spectral_moisture,
    "exposed_material": spectral_exposed,
}
used_spectral_indices = {name for names in spectral_groups.values() for name in names}
if not used_spectral_indices.issubset(hls_indices):
    raise ValueError("Every spectral-group index must also be selected as an optical index.")
if not used_spectral_indices.issubset(spectral_disturbance_direction):
    raise ValueError("Every spectral-group index needs a disturbance direction.")

hls_mask = {name: name in mask_flag_names for name in SUPPORTED_MASK_FLAGS}

# Store every user-controlled setting in one dictionary.
settings = {
    "study": {
        "output_directory": output_directory,
        "analysis_crs": analysis_crs,
        "analysis_resolution_m": analysis_resolution_m,
        "aoi_method": aoi_method,
        "aoi_path": aoi_path,
        "county_fips": county_fips,
        "tiger_year": tiger_year,
    },
    "time": {
        "baseline_years": baseline_years,
        "target_years": target_years,
        "season_start": season_start,
        "season_end": season_end,
    },
    "hls": {
        "products": hls_products,
        "maximum_granule_cloud_cover_percent": max_cloud_cover,
        "minimum_valid_observations": hls_minimum_observations,
        "indices": hls_indices,
        "mask": hls_mask,
        "reflectance_scale": reflectance_scale,
    },
    "opera_rtc": {
        "short_name": opera_short_name,
        "required_assets": opera_required_assets,
        "minimum_valid_observations": opera_minimum_observations,
        "radar_indices": radar_indices,
        "valid_mask_value": opera_valid_mask_value,
        "continuous_resampling": opera_resampling,
    },
    "nlcd": {
        "source_path": nlcd_source_path,
        "product_year": nlcd_product_year,
        "forest_classes": forest_classes,
        "categorical_resampling": nlcd_resampling,
    },
    "baseline": {
        "minimum_baseline_year_count": minimum_baseline_year_count,
        "mad_scale": mad_scale,
        "minimum_spread_floors": minimum_spread_floors,
    },
    "change": {
        "percentage_excluded_variables": percentage_excluded_variables,
        "percentage_minimum_baseline_magnitude": percentage_minimum_baseline_magnitude,
    },
    "composites": {
        "spectral_groups": spectral_groups,
        "radar_inputs": radar_inputs,
        "spectral_disturbance_direction": spectral_disturbance_direction,
        "z_cap": z_cap,
    },
    "runtime": {
        "analysis_chunk_pixels": analysis_chunk_pixels,
        "composite_block_pixels": composite_block_pixels,
        "preview_max_dimension_pixels": preview_max_dimension_pixels,
    },
    "plots": {
        "distribution_sample_size": distribution_sample_size,
        "distribution_seed": distribution_seed,
    },
    "lisa": {
        "variable": lisa_variable,
        "support_resolution_m": lisa_support_resolution_m,
        "permutations": lisa_permutations,
        "alpha": lisa_alpha,
        "weights_method": lisa_weights_method,
        "row_standardize": lisa_row_standardize,
        "minimum_valid_fraction": lisa_minimum_valid_fraction,
        "maximum_units": lisa_maximum_units,
        "random_seed": lisa_random_seed,
        "alternative": lisa_alternative,
        "n_jobs": lisa_n_jobs,
    },
    "access": {},
}

Path(output_directory).mkdir(parents=True, exist_ok=True)
tables_dir = Path(output_directory) / "tables"
config_dir = Path(output_directory) / "config"
tables_dir.mkdir(parents=True, exist_ok=True)
config_dir.mkdir(parents=True, exist_ok=True)
config_path = config_dir / "workflow_config.yaml"
config_path.write_text(yaml.safe_dump(settings, sort_keys=False))

print(f"Baseline years: {baseline_years}")
print(f"Target years: {target_years}")
print(f"Study-area method: {aoi_method}")
print(f"Configuration saved to: {config_path}")


## 3. Sign in to NASA Earthdata

Authenticate the notebook for HLS and OPERA data access.


In [ ]:
# Sign in to NASA Earthdata without saving credentials in the notebook.
# Sign in interactively without saving credentials from this notebook.
auth = earthaccess.login(strategy="interactive", persist=False)
print("signed in" if auth.authenticated else "sign-in failed")


## 4. Create the study area

Build the merged area of interest, imagery-search rectangle, and analysis grid.


In [ ]:
# only run this cell when aoi_method = "drawn"
if aoi_method == "drawn":
    draw_map = Map(center=(35.6, -82.5), zoom=7, basemap=basemaps.Esri.WorldTopoMap)
    draw_control = DrawControl(
        polygon={"shapeOptions": {"color": "#c00000", "fillOpacity": 0.1}},
        rectangle={"shapeOptions": {"color": "#c00000", "fillOpacity": 0.1}},
        circle={},
        circlemarker={},
        polyline={},
    )
    draw_map.add(draw_control)
    display(draw_map)
else:
    print("draw cell skipped")


In [ ]:
# build and save the study area and analysis grid
import urllib.request

@dataclass(frozen=True)
class GridSpec:
    crs: str
    resolution_m: float
    width: int
    height: int
    origin: tuple
    transform: Affine


def build_grid(bounds, resolution, crs):
    xmin, ymin, xmax, ymax = bounds
    width = math.ceil((xmax - xmin) / resolution)
    height = math.ceil((ymax - ymin) / resolution)
    return GridSpec(
        crs=crs,
        resolution_m=resolution,
        width=width,
        height=height,
        origin=(xmin, ymax),
        transform=from_origin(xmin, ymax, resolution, resolution),
    )


def drawn_features(control):
    data = getattr(control, "last_draw", None) or getattr(control, "data", None)
    if not data:
        raise ValueError("Draw a polygon before running this cell.")
    if isinstance(data, list):
        return gpd.GeoDataFrame.from_features(data, crs="EPSG:4326")
    if data.get("type") == "FeatureCollection":
        return gpd.GeoDataFrame.from_features(data["features"], crs="EPSG:4326")
    if data.get("type") == "Feature":
        return gpd.GeoDataFrame.from_features([data], crs="EPSG:4326")
    return gpd.GeoDataFrame(geometry=[shape(data)], crs="EPSG:4326")


if aoi_method == "fips":
    county_boundary_path = Path(output_directory) / f"tl_{tiger_year}_us_county.zip"
    if not county_boundary_path.is_file():
        url = (
            f"https://www2.census.gov/geo/tiger/TIGER{tiger_year}/COUNTY/"
            f"tl_{tiger_year}_us_county.zip"
        )
        print("downloading county boundaries")
        urllib.request.urlretrieve(url, county_boundary_path)
    counties = gpd.read_file(county_boundary_path)
    units = counties[counties["GEOID"].astype(str).isin(county_fips)].copy()
    missing = set(county_fips) - set(units["GEOID"].astype(str))
    if missing:
        raise ValueError(f"county FIPS codes not found: {sorted(missing)}")
    aoi_source_description = f"county FIPS: {','.join(county_fips)}"
elif aoi_method == "vector":
    if not Path(aoi_path).is_file():
        raise FileNotFoundError(f"study-area file not found: {aoi_path}")
    units = gpd.read_file(aoi_path)
    aoi_source_description = f"vector file: {aoi_path}"
elif aoi_method == "drawn":
    units = drawn_features(draw_control)
    aoi_source_description = "drawn polygon"
else:
    raise ValueError('aoi_method must be "fips", "vector", or "drawn"')

if units.crs is None:
    raise ValueError("the study-area features do not have a CRS")

units = units[~(units.geometry.isna() | units.geometry.is_empty)].copy()
units["geometry"] = units.geometry.apply(shapely.make_valid)
units = units[units.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].to_crs("EPSG:4326")
if units.empty:
    raise ValueError("no valid polygon features remain in the study area")

cleaned_features_4326 = units
dissolved_aoi_4326 = gpd.GeoDataFrame(
    {"name": ["study_area"]}, geometry=[units.geometry.union_all()], crs="EPSG:4326"
)
search_bbox_4326 = gpd.GeoDataFrame(
    {"name": ["search_bbox"]},
    geometry=[box(*dissolved_aoi_4326.total_bounds)],
    crs="EPSG:4326",
)

dissolved_aoi_analysis = dissolved_aoi_4326.to_crs(analysis_crs)
search_bbox_analysis = search_bbox_4326.to_crs(analysis_crs)
area_km2 = float(dissolved_aoi_analysis.area.iloc[0] / 1_000_000)
grid_spec = build_grid(
    dissolved_aoi_analysis.total_bounds,
    analysis_resolution_m,
    analysis_crs,
)

# Save the area and current configuration.
dissolved_aoi_4326.to_file(Path(output_directory) / "study_area.geojson", driver="GeoJSON")
search_bbox_4326.to_file(Path(output_directory) / "search_bbox.geojson", driver="GeoJSON")

# Save the current user configuration beside the outputs.
config_path.write_text(yaml.safe_dump(settings, sort_keys=False))

print(f"area: {area_km2:,.1f} km²")
print(f"grid: {grid_spec.width} x {grid_spec.height} pixels at {grid_spec.resolution_m} m")


In [ ]:
# review the study area before searching for imagery
# Simplify detailed outlines only for interactive map display.
_SIMPLIFY_VERTEX_THRESHOLD = 2000
_SIMPLIFY_TOLERANCE_DEGREES = 0.0001  # about 11 m at the equator


def _simplified_geojson(gdf):
    vertex_count = sum(
        len(geometry.exterior.coords) if geometry.geom_type == "Polygon" else 0
        for geometry in gdf.geometry
        if geometry is not None and not geometry.is_empty
    )
    if vertex_count > _SIMPLIFY_VERTEX_THRESHOLD:
        gdf = gdf.copy()
        gdf["geometry"] = gdf.geometry.simplify(_SIMPLIFY_TOLERANCE_DEGREES, preserve_topology=True)
    return gdf.__geo_interface__


west, south, east, north = search_bbox_4326.total_bounds
map_center = ((south + north) / 2, (west + east) / 2)

aoi_review_map = Map(center=map_center, zoom=10, basemap=basemaps.Esri.WorldTopoMap)
aoi_review_map.add(
    GeoJSON(
        data=_simplified_geojson(dissolved_aoi_4326),
        name="Area of interest (exact outline, used for the final result)",
        style={"color": "#1f4e79", "fillOpacity": 0.05, "weight": 2},
    )
)
aoi_review_map.add(
    GeoJSON(
        data=_simplified_geojson(search_bbox_4326),
        name="Search rectangle (search and first trim only, not the final outline)",
        style={"color": "#c00000", "fillOpacity": 0.05, "weight": 2},
    )
)
aoi_review_map.add(LayersControl())
display(aoi_review_map)


## 5. Search for imagery

Find HLS and OPERA RTC-S1 scenes for the selected area, years, and season.


In [ ]:
# search HLS and OPERA RTC-S1 imagery
@dataclass
class GranuleRecord:
    """Store the metadata and asset URLs for one NASA granule."""
    granule_id: str
    short_name: str
    provider: str
    acquisition_datetime: str
    processing_datetime: str
    cloud_cover_percent: float
    asset_urls: dict = field(default_factory=dict)


def build_year_season_windows(years, season_start, season_end):
    """Build one seasonal date range for each requested year."""
    # Build a separate seasonal window for each year.
    windows = []
    for year in years:
        start = f"{year}-{season_start}T00:00:00Z"
        end = f"{year}-{season_end}T23:59:59Z"
        windows.append((start, end))
    return windows


def _extract_cloud_cover(umm):
    # OPERA RTC-S1 records generally do not report this; None is expected for those.
    for attribute in umm.get("AdditionalAttributes", []):
        if attribute.get("Name") in ("CLOUD_COVERAGE", "CLOUD_COVER"):
            values = attribute.get("Values") or []
            if values:
                try:
                    return float(values[0])
                except (TypeError, ValueError):
                    return None
    return None


def _extract_processing_datetime(umm):
    for provider_date in umm.get("ProviderDates", []):
        if provider_date.get("Type") in ("Insert", "Update"):
            return provider_date.get("Date")
    return None


def _extract_asset_urls(umm):
    urls = {}
    for related_url in umm.get("RelatedUrls", []):
        if related_url.get("Type") == "GET DATA":
            url = related_url.get("URL")
            if url:
                urls[Path(urlparse(url).path).name] = url
    return urls


def parse_cmr_granule(granule, short_name, provider):
    """Convert one Earthaccess search result into a GranuleRecord."""
    umm = granule["umm"]
    return GranuleRecord(
        granule_id=umm["GranuleUR"],
        short_name=short_name,
        provider=provider,
        acquisition_datetime=umm["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"],
        processing_datetime=_extract_processing_datetime(umm),
        cloud_cover_percent=_extract_cloud_cover(umm),
        asset_urls=_extract_asset_urls(umm),
    )


def search_granules_for_window(short_name, provider, bbox_4326, temporal_window, search_fn):
    """Search one NASA collection for one date window."""
    raw_granules = search_fn(
        short_name=short_name, bounding_box=bbox_4326, temporal=temporal_window
    )
    return [parse_cmr_granule(granule, short_name, provider) for granule in raw_granules]


def search_hls_granules(
    products, bbox_4326, years, season_start, season_end, search_fn, max_cloud_percent
):
    """Search the configured HLS collections for all requested years."""
    windows = build_year_season_windows(years, season_start, season_end)
    records = []
    for product in products:
        for window in windows:
            records.extend(
                search_granules_for_window(product, "LPDAAC", bbox_4326, window, search_fn)
            )
    # Filtered here rather than by the search service, since the cloud-cover field is not
    # reported consistently for every collection. A picture with no reported cloud cover is kept.
    return [
        record
        for record in records
        if record.cloud_cover_percent is None or record.cloud_cover_percent <= max_cloud_percent
    ]


def search_opera_rtc_granules(short_name, bbox_4326, years, season_start, season_end, search_fn):
    """Search OPERA RTC-S1 for all requested years."""
    windows = build_year_season_windows(years, season_start, season_end)
    records = []
    for window in windows:
        records.extend(search_granules_for_window(short_name, "ASF", bbox_4326, window, search_fn))
    return records

searches_dir = Path(output_directory) / "searches"
searches_dir.mkdir(parents=True, exist_ok=True)
search_years = list(baseline_years) + list(target_years)

hls_granules = search_hls_granules(
    settings["hls"]["products"],
    tuple(search_bbox_4326.total_bounds),
    search_years,
    season_start,
    season_end,
    earthaccess.search_data,
    max_cloud_cover,
)
(searches_dir / "hls_granules.json").write_text(
    json.dumps([granule.__dict__ for granule in hls_granules], indent=2)
)
print(f"found {len(hls_granules)} HLS pictures")

opera_granules = search_opera_rtc_granules(
    settings["opera_rtc"]["short_name"],
    tuple(search_bbox_4326.total_bounds),
    search_years,
    season_start,
    season_end,
    earthaccess.search_data,
)
(searches_dir / "opera_rtc_granules.json").write_text(
    json.dumps([granule.__dict__ for granule in opera_granules], indent=2)
)
print(f"found {len(opera_granules)} OPERA RTC-S1 pictures")


In [ ]:
# summarize the imagery found for each product
granule_rows = []
for product_label, granules_filename in (
    ("HLS", "hls_granules.json"),
    ("OPERA RTC-S1", "opera_rtc_granules.json"),
):
    granules_path = searches_dir / granules_filename
    if not granules_path.is_file():
        continue
    for granule in json.loads(granules_path.read_text()):
        granule_rows.append(
            {
                "product": product_label,
                "short_name": granule["short_name"],
                "acquisition_date": granule["acquisition_datetime"][:10],
                "cloud_cover_percent": granule["cloud_cover_percent"],
            }
        )

granules_df = pd.DataFrame(granule_rows)
if not granules_df.empty:
    coverage = (
        granules_df.groupby(["product", "short_name"]).size().rename("scene_count").reset_index()
    )
    display(coverage)
else:
    print("no pictures found yet, run the search cell above first")


## 6. Choose streaming or downloading

Select how HLS and OPERA files are accessed and whether this is a pilot(test run) or full run.


In [ ]:
# Ask how imagery should be accessed at the point where access begins.
hls_access_choice = ask_choice(
    "HLS access method",
    choices=("stream", "download", "auto"),
    default="auto",
)
opera_access_choice = ask_choice(
    "OPERA access method",
    choices=("stream", "download", "auto"),
    default="auto",
)
run_size_choice = ask_choice(
    "Run size",
    choices=("pilot", "full"),
    default="pilot",
)
pilot_max_scenes_per_product_year = ask(
    "Pilot maximum scenes per product and year",
    default=3,
    parser=int,
    validate=lambda value: value >= 1,
)
scene_workers = ask(
    "Parallel scene workers",
    default=2,
    parser=int,
    validate=lambda value: value >= 1,
)
download_threads = ask(
    "Download threads",
    default=4,
    parser=int,
    validate=lambda value: value >= 1,
)
auto_benchmark_scene_count = ask(
    "Scenes used to test automatic access",
    default=1,
    parser=int,
    validate=lambda value: value >= 1,
)

settings["access"] = {
    "hls_requested_mode": hls_access_choice,
    "opera_requested_mode": opera_access_choice,
    "auto_benchmark_scene_count": auto_benchmark_scene_count,
}
settings["runtime"].update({
    "run_size": run_size_choice,
    "pilot_max_scenes_per_product_year": pilot_max_scenes_per_product_year,
    "scene_workers": scene_workers,
    "download_threads": download_threads,
})
config_path.write_text(yaml.safe_dump(settings, sort_keys=False))

print(f"HLS access requested: {hls_access_choice}")
print(f"OPERA access requested: {opera_access_choice}")
print(f"Run size: {run_size_choice}")


In [ ]:
# raster alignment, windowing, clipping, and GeoTIFF helpers
# Read and align raster bands in bounded windows.
CATEGORICAL_RESAMPLING = {"nearest": Resampling.nearest, "mode": Resampling.mode}
CONTINUOUS_RESAMPLING = {
    "nearest": Resampling.nearest,
    "bilinear": Resampling.bilinear,
    "cubic": Resampling.cubic,
    "average": Resampling.average,
}


def resampling_from_name(name, categorical):
    """Return the rasterio resampling method for a configured name."""
    table = CATEGORICAL_RESAMPLING if categorical else CONTINUOUS_RESAMPLING
    if name not in table:
        raise ValueError(f"{name!r} is not a supported resampling method")
    return table[name]


def raster_matches_grid(path, grid_spec):
    """True only if width, height, pixel grid, AND projection all match a shared file being open is not enough."""
    try:
        with rasterio.open(path) as source:
            return (
                source.width == grid_spec.width
                and source.height == grid_spec.height
                and source.transform == grid_spec.transform
                and source.crs == RasterioCRS.from_string(grid_spec.crs)
            )
    except Exception:
        return False


def open_and_align_band(path, grid_spec, resampling, band_index=1):
    """Open one raster band and align it to the analysis grid."""
    # WarpedVRT reads only the source pixels needed for the target grid.
    with (
        rasterio.open(path) as source,
        WarpedVRT(
            source,
            crs=grid_spec.crs,
            transform=grid_spec.transform,
            width=grid_spec.width,
            height=grid_spec.height,
            resampling=resampling,
        ) as vrt,
    ):
        return vrt.read(band_index)


def iter_grid_windows(grid_spec, block_pixels):
    """Bounded, non-overlapping pixel windows covering the whole grid, read left-to-right, top-to-bottom."""
    windows = []
    for row_start in range(0, grid_spec.height, block_pixels):
        row_size = min(block_pixels, grid_spec.height - row_start)
        for col_start in range(0, grid_spec.width, block_pixels):
            col_size = min(block_pixels, grid_spec.width - col_start)
            windows.append(Window(col_start, row_start, col_size, row_size))
    return windows


def rasterize_aoi_mask(aoi_geometry, grid_spec):
    """True everywhere inside the exact area outline, for the final trim (not the search rectangle)."""
    mask = rasterize(
        [(aoi_geometry, 1)],
        out_shape=(grid_spec.height, grid_spec.width),
        transform=grid_spec.transform,
        fill=0,
    )
    return mask.astype(bool)


def clip_to_aoi(array, aoi_mask, fill_value=np.nan):
    """Set pixels outside the study area to NoData."""
    result = array.copy()
    result[..., ~aoi_mask] = fill_value
    return result


def write_geotiff(
    path, data, grid_spec, band_names=None, nodata=np.nan, dtype="float32", tags=None
):
    """data may be one band (2D) or several bands stacked together (3D)."""
    array = np.asarray(data)
    if array.ndim == 2:
        array = array[np.newaxis, ...]
    destination_path = Path(path)
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    profile = {
        "driver": "GTiff",
        "height": grid_spec.height,
        "width": grid_spec.width,
        "count": array.shape[0],
        "dtype": dtype,
        "crs": grid_spec.crs,
        "transform": grid_spec.transform,
        "nodata": nodata,
    }
    with rasterio.open(destination_path, "w", **profile) as destination:
        destination.write(array.astype(dtype))
        if band_names is not None:
            for band_number, name in enumerate(band_names, start=1):
                destination.set_band_description(band_number, name)
        if tags:
            destination.update_tags(**tags)
    return destination_path


def read_window(path, window, band_indexes=None):
    """Read one raster window as float32."""
    with rasterio.open(path) as source:
        indexes = band_indexes or list(range(1, source.count + 1))
        return source.read(indexes, window=window)


def read_full_array(path, band_indexes=None):
    """Read the first raster band as float32."""
    # This is only used to read one already-composited, already-small file at a time, never a
    # full scene stack.
    with rasterio.open(path) as source:
        indexes = band_indexes or list(range(1, source.count + 1))
        return source.read(indexes)


In [ ]:
# stream or download imagery and compare access methods
# Resolve remote assets to streams or validated local downloads.
DEFAULT_MINIMUM_ASSET_BYTES = 1024


class AccessError(Exception):
    """Report an imagery streaming or download failure."""
    pass


def download_directory(output_directory, product):
    """Create and return the local download folder for a product."""
    return Path(output_directory) / "downloads" / product


def cache_directory(output_directory, product):
    """Where the finished, cleaned, index-calculated picture is cached, for both access modes."""
    return Path(output_directory) / "cache" / product


def raster_asset_is_valid(path, min_size_bytes=DEFAULT_MINIMUM_ASSET_BYTES, expected_bands=None):
    """Check that a local raster exists, is large enough, and opens correctly."""
    resolved_path = Path(path)
    if not resolved_path.is_file():
        return False
    try:
        if resolved_path.stat().st_size < min_size_bytes:
            return False
        with rasterio.open(resolved_path) as source:
            if expected_bands is not None and source.count != expected_bands:
                return False
    except Exception:
        return False
    return True


def build_local_asset_index(root):
    """Index valid local raster files by filename."""
    # Indexes the whole download folder, not just one year's folder, so a file already fetched
    # for one year satisfies another year's request too.
    index = {}
    root = Path(root)
    if root.exists():
        for path in root.rglob("*"):
            if path.is_file():
                index.setdefault(path.name, path)
    return index


def find_valid_local_asset(directory, filename, index, validator=raster_asset_is_valid):
    """Find a valid downloaded asset matching a remote URL."""
    directory = Path(directory)
    candidates = [directory / filename]
    indexed = index.get(filename)
    if indexed is not None:
        candidates.append(indexed)
    for candidate in candidates:
        if candidate.exists() and validator(candidate):
            return candidate
    return None


def resolve_asset_source(
    url, access_mode, download_directory, local_index, open_stream_fn, fetch_fn, download_threads
):
    """Return a local path for downloads or an open remote handle for streaming."""
    del download_threads
    if access_mode == "download":
        filename = Path(urlparse(url).path).name
        found = find_valid_local_asset(download_directory, filename, local_index)
        if found is not None:
            return found
        destination = Path(download_directory) / filename
        destination.parent.mkdir(parents=True, exist_ok=True)
        fetch_fn(url, destination)
        if not raster_asset_is_valid(destination):
            raise AccessError(f"downloaded file failed validation: {destination}")
        return destination
    if access_mode == "stream":
        return open_stream_fn(url)
    raise ValueError(f"unsupported access mode: {access_mode!r}")


def close_all_quietly(handles):
    """Close open handles without hiding the original processing error."""
    for handle in handles:
        with contextlib.suppress(Exception):
            handle.close()

def benchmark_access_modes(stream_sample_fn, download_sample_fn):
    """Time a small real sample of each option; whichever is faster is recommended."""
    start = time.perf_counter()
    stream_sample_fn()
    stream_elapsed = time.perf_counter() - start

    start = time.perf_counter()
    download_sample_fn()
    download_elapsed = time.perf_counter() - start

    if stream_elapsed <= download_elapsed:
        reason = f"streaming ({stream_elapsed:.3f}s) was at least as fast as downloading ({download_elapsed:.3f}s)"
        return "stream", reason
    reason = (
        f"downloading ({download_elapsed:.3f}s) was faster than streaming ({stream_elapsed:.3f}s)"
    )
    return "download", reason


def select_access_mode(requested_mode, stream_sample_fn=None, download_sample_fn=None):
    """'stream' and 'download' are honored exactly as asked. 'auto' times a small real sample of each."""
    if requested_mode in ("stream", "download"):
        return requested_mode, "this access mode was requested directly"
    return benchmark_access_modes(stream_sample_fn, download_sample_fn)


In [ ]:
# Select the final access methods and identify the required image assets.
max_scenes_per_product_year = (
    settings["runtime"]["pilot_max_scenes_per_product_year"]
    if run_size_choice == "pilot"
    else 10_000
)


def _sample_asset_urls(granules, count):
    urls = []
    for granule in granules[:count]:
        urls.extend(granule.asset_urls.values())
    return urls


def _open_stream_fn(url):
    return earthaccess.open([url])[0]


def _fetch_fn(url, destination):
    earthaccess.download([url], str(destination.parent))


def _stream_sample_fn(granules, count):
    for url in _sample_asset_urls(granules, count):
        handle = earthaccess.open([url])[0]
        handle.read(1024)
        handle.close()


def _download_sample_fn(granules, count, product_name):
    urls = _sample_asset_urls(granules, count)
    target_dir = download_directory(output_directory, f"{product_name}_benchmark_sample")
    earthaccess.download(urls, str(target_dir))


benchmark_sample_size = settings["access"]["auto_benchmark_scene_count"]
hls_selected_mode, hls_reason = select_access_mode(
    hls_access_choice,
    lambda: _stream_sample_fn(hls_granules, benchmark_sample_size),
    lambda: _download_sample_fn(hls_granules, benchmark_sample_size, "hls"),
)
print(f"HLS access: requested={hls_access_choice}, selected={hls_selected_mode} ({hls_reason})")

opera_selected_mode, opera_reason = select_access_mode(
    opera_access_choice,
    lambda: _stream_sample_fn(opera_granules, benchmark_sample_size),
    lambda: _download_sample_fn(opera_granules, benchmark_sample_size, "opera_rtc"),
)
print(
    f"OPERA RTC-S1 access: requested={opera_access_choice}, selected={opera_selected_mode} ({opera_reason})"
)

pd.DataFrame(
    [
        {
            "product": "hls",
            "requested_mode": hls_access_choice,
            "selected_mode": hls_selected_mode,
            "reason": hls_reason,
        },
        {
            "product": "opera_rtc",
            "requested_mode": opera_access_choice,
            "selected_mode": opera_selected_mode,
            "reason": opera_reason,
        },
    ]
).to_csv(Path(output_directory) / "tables" / "access_benchmark.csv", index=False, mode="a")

# Map each HLS product to its optical and Fmask asset codes.
HLS_BAND_CODES = {
    "HLSL30": {
        "blue": "B02",
        "green": "B03",
        "red": "B04",
        "nir": "B05",
        "swir1": "B06",
        "swir2": "B07",
        "fmask": "Fmask",
    },
    "HLSS30": {
        "blue": "B02",
        "green": "B03",
        "red": "B04",
        "nir": "B8A",
        "swir1": "B11",
        "swir2": "B12",
        "fmask": "Fmask",
    },
}
REQUIRED_HLS_BANDS = ("blue", "green", "red", "nir", "swir1", "swir2", "fmask")

# Match OPERA asset filenames to VV, VH, and validity-mask bands.
OPERA_ASSET_CODES = {"VV": "VV", "VH": "VH", "mask": "mask"}


def resolve_hls_band_urls(granule):
    """Match an HLS granule to the required optical and Fmask assets."""
    band_codes = HLS_BAND_CODES.get(granule.short_name)
    if band_codes is None:
        raise ValueError(f"unknown HLS product type: {granule.short_name!r}")
    resolved = {}
    for semantic_name in REQUIRED_HLS_BANDS:
        code = band_codes[semantic_name]
        match = next(
            (url for filename, url in granule.asset_urls.items() if f".{code}." in filename), None
        )
        if match is None:
            raise ValueError(f"could not find band {semantic_name!r} for {granule.granule_id!r}")
        resolved[semantic_name] = match
    return resolved


def resolve_opera_asset_urls(granule, required_assets):
    """Match an OPERA granule to VV, VH, and validity-mask assets."""
    resolved = {}
    for asset_name in required_assets:
        code = OPERA_ASSET_CODES[asset_name]
        match = next(
            (url for filename, url in granule.asset_urls.items() if f"_{code}." in filename), None
        )
        if match is None:
            raise ValueError(f"could not find asset {asset_name!r} for {granule.granule_id!r}")
        resolved[asset_name] = match
    return resolved


## 7. Process the imagery

Align, mask, calculate indices, and cache each HLS and OPERA scene.


In [ ]:
# mask clouds, aerosols, shadows, snow, and invalid radar pixels
def hls_fmask_clean(fmask, mask_flags):
    """Return True for HLS pixels that pass the configured Fmask checks."""
    values = np.where(np.isfinite(fmask), fmask, 255).astype("uint16")
    bad = np.zeros(values.shape, dtype=bool)
    if mask_flags["cirrus"]:
        bad |= ((values >> 0) & 1).astype(bool)
    if mask_flags["cloud"]:
        bad |= ((values >> 1) & 1).astype(bool)
    if mask_flags["adjacent_cloud"]:
        bad |= ((values >> 2) & 1).astype(bool)
    if mask_flags["cloud_shadow"]:
        bad |= ((values >> 3) & 1).astype(bool)
    if mask_flags["snow"]:
        bad |= ((values >> 4) & 1).astype(bool)
    if mask_flags["high_aerosol"]:
        bad |= ((values >> 6) & 3) == 3
    return ~bad


def opera_validity_mask(mask, valid_mask_value):
    """Return True where OPERA pixels match the configured valid value."""
    return mask == valid_mask_value


def apply_opera_validity(vv, vh, mask, valid_mask_value):
    """Sets VV/VH to a missing-value marker everywhere the quality-check layer marks a pixel bad.
    This must run before any radar index math."""
    valid = opera_validity_mask(mask, valid_mask_value)
    vv_masked = np.where(valid, vv, np.nan).astype(vv.dtype, copy=False)
    vh_masked = np.where(valid, vh, np.nan).astype(vh.dtype, copy=False)
    return vv_masked, vh_masked

hls_mask_flags = settings["hls"]["mask"]
opera_valid_mask_value = settings["opera_rtc"]["valid_mask_value"]
print("Pixel masks are ready.")


In [ ]:
# define the optical and radar indices
# Keep a consistent output band order.
OPTICAL_INDEX_NAMES = ("ndvi", "evi", "nbr", "ndmi", "dfi", "ndti")
RADAR_INDEX_NAMES = ("vv_db", "vh_db", "rvi", "mrdfi", "vh_vv_ratio", "dprvi")
RATIO_EPSILON = np.float32(1e-6)


def _ratio(numerator, denominator, eps):
    # Below this very small size, a ratio is left blank rather than reported as a huge or
    # divide-by-zero number.
    output = np.full(numerator.shape, np.nan, dtype="float32")
    valid = np.isfinite(numerator) & np.isfinite(denominator) & (np.abs(denominator) > eps)
    output[valid] = numerator[valid] / denominator[valid]
    return output


def calculate_optical_indices(
    green, red, nir, swir1, swir2, blue, fmask, mask_flags, reflectance_scale
):
    """Calculate the configured optical indices from aligned HLS bands."""
    # Convert HLS scaled integers to reflectance.
    bands = [green, red, nir, swir1, swir2, blue]
    scale = np.float32(reflectance_scale)
    green, red, nir, swir1, swir2, blue = [
        np.asarray(band, dtype="float32") * scale for band in bands
    ]
    clean = hls_fmask_clean(fmask, mask_flags)

    ndvi = _ratio(nir - red, nir + red, RATIO_EPSILON)
    evi = _ratio(2.5 * (nir - red), nir + 6.0 * red - 7.5 * blue + 1.0, RATIO_EPSILON)
    nbr = _ratio(nir - swir2, nir + swir2, RATIO_EPSILON)
    ndmi = _ratio(nir - swir1, nir + swir1, RATIO_EPSILON)
    dfi = 100.0 * (1.0 - _ratio(swir2, swir1, RATIO_EPSILON)) * _ratio(red, nir, RATIO_EPSILON)
    ndti = _ratio(swir1 - swir2, swir1 + swir2, RATIO_EPSILON)

    indices = {"ndvi": ndvi, "evi": evi, "nbr": nbr, "ndmi": ndmi, "dfi": dfi, "ndti": ndti}
    for array in indices.values():
        array[~clean] = np.nan
    return indices

BACKSCATTER_EPSILON = np.float32(1e-8)


def calculate_radar_indices(vv, vh):
    """vv/vh must already be masked (missing-value marked) for bad pixels."""
    vv = np.asarray(vv, dtype="float32")
    vh = np.asarray(vh, dtype="float32")
    valid = (
        np.isfinite(vv) & np.isfinite(vh) & (vv > BACKSCATTER_EPSILON) & (vh > BACKSCATTER_EPSILON)
    )

    outputs = {name: np.full(vv.shape, np.nan, dtype="float32") for name in RADAR_INDEX_NAMES}
    if not valid.any():
        return outputs

    vv_valid = vv[valid]
    vh_valid = vh[valid]
    total = vv_valid + vh_valid
    ratio = vh_valid / vv_valid

    outputs["vv_db"][valid] = 10.0 * np.log10(vv_valid)
    outputs["vh_db"][valid] = 10.0 * np.log10(vh_valid)
    outputs["rvi"][valid] = 4.0 * vh_valid / total
    outputs["mrdfi"][valid] = (vv_valid - vh_valid) / total
    outputs["vh_vv_ratio"][valid] = ratio
    outputs["dprvi"][valid] = ratio * (ratio + 3.0) / ((ratio + 1.0) ** 2)
    return outputs


In [ ]:
# calculate and cache HLS indices for each scene
# Process and cache one HLS scene at a time.
cache_hls_dir = cache_directory(output_directory, "hls")
download_hls_dir = download_directory(output_directory, "hls")
local_hls_index = build_local_asset_index(download_hls_dir)


# Reuse valid cached output before reprocessing.
def process_one_hls_granule(granule):
    """Align, mask, calculate, and cache indices for one HLS granule."""
    cache_path = cache_hls_dir / f"{granule.granule_id}.tif"
    if raster_asset_is_valid(cache_path, expected_bands=len(settings["hls"]["indices"])):
        return cache_path
    band_urls = resolve_hls_band_urls(granule)
    sources = {
        name: resolve_asset_source(
            url,
            hls_selected_mode,
            download_hls_dir,
            local_hls_index,
            _open_stream_fn,
            _fetch_fn,
            download_threads,
        )
        for name, url in band_urls.items()
    }
    try:
        aligned = {
            name: open_and_align_band(sources[name], grid_spec, Resampling.bilinear)
            for name in ("blue", "green", "red", "nir", "swir1", "swir2")
        }
        fmask = open_and_align_band(sources["fmask"], grid_spec, Resampling.nearest)
        indices = calculate_optical_indices(
            aligned["green"],
            aligned["red"],
            aligned["nir"],
            aligned["swir1"],
            aligned["swir2"],
            aligned["blue"],
            fmask,
            hls_mask_flags,
            settings["hls"]["reflectance_scale"],
        )
        band_order = [name for name in OPTICAL_INDEX_NAMES if name in settings["hls"]["indices"]]
        stacked = np.stack([indices[name] for name in band_order], axis=0)
        write_geotiff(
            cache_path,
            stacked,
            grid_spec,
            band_names=band_order,
            tags={"granule_id": granule.granule_id, "provider": granule.provider},
        )
    finally:
        close_all_quietly(list(sources.values()))
    return cache_path


def _year_of(granule):
    return int(granule.acquisition_datetime[:4])


def _capped_by_year(granules, cap):
    by_year = {}
    for granule in granules:
        by_year.setdefault(_year_of(granule), []).append(granule)
    return {year: scenes[:cap] for year, scenes in by_year.items()}


hls_by_year = _capped_by_year(hls_granules, max_scenes_per_product_year)
hls_cache_paths_by_year = {}
for year, year_granules in sorted(hls_by_year.items()):
    cache_paths = []
    with ThreadPoolExecutor(max_workers=max(1, scene_workers)) as pool:
        futures = {
            pool.submit(process_one_hls_granule, granule): granule for granule in year_granules
        }
        for future in as_completed(futures):
            granule = futures[future]
            try:
                cache_paths.append(future.result())
            except Exception as exc:
                print(f"hls: picture {granule.granule_id} failed to process ({exc})")
    hls_cache_paths_by_year[year] = cache_paths
    print(f"hls {year}: {len(cache_paths)} usable picture(s) cached")


In [ ]:
# calculate and cache OPERA RTC-S1 indices for each scene
cache_opera_dir = cache_directory(output_directory, "opera_rtc")
download_opera_dir = download_directory(output_directory, "opera_rtc")
local_opera_index = build_local_asset_index(download_opera_dir)


# Same cache-first pattern as process_one_hls_granule, applied to the OPERA RTC-S1 bands.
def process_one_opera_granule(granule):
    """Align, mask, calculate, and cache indices for one OPERA granule."""
    cache_path = cache_opera_dir / f"{granule.granule_id}.tif"
    if raster_asset_is_valid(
        cache_path, expected_bands=len(settings["opera_rtc"]["radar_indices"])
    ):
        return cache_path
    asset_urls = resolve_opera_asset_urls(granule, settings["opera_rtc"]["required_assets"])
    sources = {
        name: resolve_asset_source(
            url,
            opera_selected_mode,
            download_opera_dir,
            local_opera_index,
            _open_stream_fn,
            _fetch_fn,
            download_threads,
        )
        for name, url in asset_urls.items()
    }
    try:
        continuous_resampling = resampling_from_name(
            settings["opera_rtc"]["continuous_resampling"], categorical=False
        )
        vv = open_and_align_band(sources["VV"], grid_spec, continuous_resampling)
        vh = open_and_align_band(sources["VH"], grid_spec, continuous_resampling)
        mask = open_and_align_band(sources["mask"], grid_spec, Resampling.nearest)
        vv_masked, vh_masked = apply_opera_validity(
            vv, vh, mask, settings["opera_rtc"]["valid_mask_value"]
        )
        indices = calculate_radar_indices(vv_masked, vh_masked)
        band_order = [
            name for name in RADAR_INDEX_NAMES if name in settings["opera_rtc"]["radar_indices"]
        ]
        stacked = np.stack([indices[name] for name in band_order], axis=0)
        write_geotiff(
            cache_path,
            stacked,
            grid_spec,
            band_names=band_order,
            tags={"granule_id": granule.granule_id, "provider": granule.provider},
        )
    finally:
        close_all_quietly(list(sources.values()))
    return cache_path


opera_by_year = _capped_by_year(opera_granules, max_scenes_per_product_year)
opera_cache_paths_by_year = {}
for year, year_granules in sorted(opera_by_year.items()):
    cache_paths = []
    with ThreadPoolExecutor(max_workers=max(1, scene_workers)) as pool:
        futures = {
            pool.submit(process_one_opera_granule, granule): granule for granule in year_granules
        }
        for future in as_completed(futures):
            granule = futures[future]
            try:
                cache_paths.append(future.result())
            except Exception as exc:
                print(f"opera-rtc: picture {granule.granule_id} failed to process ({exc})")
    opera_cache_paths_by_year[year] = cache_paths
    print(f"opera-rtc {year}: {len(cache_paths)} usable picture(s) cached")


## 8. Build the forest mask and annual composites

Limit the analysis to forest and combine valid scenes into one composite per year.


In [ ]:
# align NLCD and keep forest classes
if not nlcd_source_path or not Path(nlcd_source_path).is_file():
    raise FileNotFoundError(
        "Set nlcd_source_path in the configuration cell to a local NLCD raster."
    )

settings["nlcd"]["source_path"] = nlcd_source_path
settings["nlcd"]["product_year"] = nlcd_product_year
config_path.write_text(yaml.safe_dump(settings, sort_keys=False))

def forest_membership(nlcd_classes, forest_classes):
    """Return True where NLCD classes are configured as forest."""
    return np.isin(nlcd_classes, forest_classes)


def load_forest_mask(
    nlcd_source_path, forest_classes, categorical_resampling, grid_spec, aoi_geometry
):
    """Align NLCD to the analysis grid and create a binary forest mask."""
    resampling = resampling_from_name(categorical_resampling, categorical=True)
    # Preserve categorical NLCD class values.
    aligned_classes = open_and_align_band(nlcd_source_path, grid_spec, resampling)
    forest = forest_membership(aligned_classes, forest_classes)
    aoi_mask = rasterize_aoi_mask(aoi_geometry, grid_spec)
    return clip_to_aoi(forest, aoi_mask, fill_value=False)


forest_mask = load_forest_mask(
    settings["nlcd"]["source_path"],
    tuple(settings["nlcd"]["forest_classes"]),
    settings["nlcd"]["categorical_resampling"],
    grid_spec,
    dissolved_aoi_analysis.geometry.iloc[0],
)
forest_mask_path = Path(output_directory) / "composites" / "nlcd_forest_mask.tif"
write_geotiff(
    forest_mask_path,
    forest_mask.astype("uint8"),
    grid_spec,
    band_names=["forest"],
    nodata=None,
    dtype="uint8",
    tags={"nlcd_product_year": str(settings["nlcd"]["product_year"])},
)
print(f"forest mask: {int(forest_mask.sum())} of {forest_mask.size} pixels are forest")


In [ ]:
# create annual median composites
def annual_median_composite_block(scene_stack, minimum_observations):
    """Calculate a median composite and valid-scene count for one block."""
    # scene_stack shape: (scene, band, row, column).
    complete_valid = np.all(np.isfinite(scene_stack), axis=1)
    complete_count = complete_valid.sum(axis=0).astype("uint16")
    output = np.nanmedian(scene_stack, axis=0).astype("float32")
    output[:, complete_count < int(minimum_observations)] = np.nan
    return output, complete_count


# Build the annual composite one window at a time.
def compute_annual_composite(
    scene_paths,
    band_names,
    grid_spec,
    minimum_observations,
    composite_output_path,
    count_output_path,
    block_pixels,
):
    """Build one annual median composite with windowed raster processing."""
    for path in scene_paths:
        if not raster_matches_grid(path, grid_spec):
            raise ValueError(
                f"picture {path} is not aligned to the shared grid; every cached picture must already match it"
            )

    composite_path = Path(composite_output_path)
    count_path = Path(count_output_path)
    composite_path.parent.mkdir(parents=True, exist_ok=True)
    count_path.parent.mkdir(parents=True, exist_ok=True)
    composite_profile = {
        "driver": "GTiff",
        "height": grid_spec.height,
        "width": grid_spec.width,
        "count": len(band_names),
        "dtype": "float32",
        "crs": grid_spec.crs,
        "transform": grid_spec.transform,
        "nodata": np.nan,
    }
    count_profile = dict(composite_profile, count=1, dtype="uint16", nodata=0)

    windows = iter_grid_windows(grid_spec, block_pixels)
    with (
        rasterio.open(composite_path, "w", **composite_profile) as composite_dst,
        rasterio.open(count_path, "w", **count_profile) as count_dst,
    ):
        for band_number, name in enumerate(band_names, start=1):
            composite_dst.set_band_description(band_number, name)
        for window in windows:
            # A bounded per-window stack: (picture, band, window_rows, window_cols).
            window_scenes = [read_window(path, window) for path in scene_paths]
            scene_stack = np.stack(window_scenes, axis=0).astype("float32")
            composite_block, count_block = annual_median_composite_block(
                scene_stack, minimum_observations
            )
            composite_dst.write(composite_block, window=window)
            count_dst.write(count_block[np.newaxis, ...], window=window)
            del (
                window_scenes,
                scene_stack,
                composite_block,
                count_block,
            )  # release before the next window
    return composite_path, count_path

composites_dir = Path(output_directory) / "composites"
for year, cache_paths in sorted(hls_cache_paths_by_year.items()):
    if not cache_paths:
        continue
    compute_annual_composite(
        cache_paths,
        list(settings["hls"]["indices"]),
        grid_spec,
        settings["hls"]["minimum_valid_observations"],
        composites_dir / f"hls_{year}.tif",
        composites_dir / f"hls_{year}_count.tif",
        settings["runtime"]["composite_block_pixels"],
    )
    print(f"hls {year}: wrote the yearly average from {len(cache_paths)} picture(s)")

for year, cache_paths in sorted(opera_cache_paths_by_year.items()):
    if not cache_paths:
        continue
    compute_annual_composite(
        cache_paths,
        list(settings["opera_rtc"]["radar_indices"]),
        grid_spec,
        settings["opera_rtc"]["minimum_valid_observations"],
        composites_dir / f"opera_rtc_{year}.tif",
        composites_dir / f"opera_rtc_{year}_count.tif",
        settings["runtime"]["composite_block_pixels"],
    )
    print(f"opera-rtc {year}: wrote the yearly average from {len(cache_paths)} picture(s)")


In [ ]:
# create small previews for notebook maps and figures
# Use reduced previews for all notebook displays.
def compute_preview_shape(height, width, max_dimension_pixels):
    """Scale raster dimensions to a bounded preview size."""
    if height <= max_dimension_pixels and width <= max_dimension_pixels:
        return int(height), int(width)
    scale = max_dimension_pixels / max(height, width)
    return max(1, round(height * scale)), max(1, round(width * scale))


@dataclass
class PreviewArtifact:
    """Store a preview array and the metadata needed to map it."""
    array: np.ndarray  # one band, shape (out_height, out_width)
    grid_spec: GridSpec
    categorical: bool


def build_preview(raster_path, max_dimension_pixels, categorical=False, band_index=1):
    """Read a reduced continuous or categorical preview from a raster."""
    # Preserve classes for categorical previews and average continuous previews.
    resampling = Resampling.nearest if categorical else Resampling.average
    with rasterio.open(raster_path) as source:
        out_height, out_width = compute_preview_shape(
            source.height, source.width, max_dimension_pixels
        )
        array = source.read(band_index, out_shape=(out_height, out_width), resampling=resampling)
        decimated_transform = source.transform * Affine.scale(
            source.width / out_width, source.height / out_height
        )
        preview_spec = GridSpec(
            crs=str(source.crs),
            resolution_m=source.res[0] * (source.width / out_width),
            width=out_width,
            height=out_height,
            origin=(source.bounds.left, source.bounds.top),
            transform=decimated_transform,
        )
    return PreviewArtifact(array=array, grid_spec=preview_spec, categorical=categorical)


def preview_bounds_4326(artifact):
    """(west, south, east, north) in plain latitude/longitude, what ipyleaflet needs for a map overlay."""
    from pyproj import Transformer
    from rasterio.transform import array_bounds

    left, bottom, right, top = array_bounds(
        artifact.grid_spec.height, artifact.grid_spec.width, artifact.grid_spec.transform
    )
    if artifact.grid_spec.crs in ("EPSG:4326", "urn:ogc:def:crs:EPSG::4326"):
        return (left, bottom, right, top)
    transformer = Transformer.from_crs(artifact.grid_spec.crs, "EPSG:4326", always_xy=True)
    return transformer.transform_bounds(left, bottom, right, top)


# Renders one preview array to a small transparent PNG (blank pixels stay transparent) sized to
# match the array, for use as an ipyleaflet image overlay.
def preview_to_png(artifact, output_path, colormap, value_range):
    """Render a preview array as a transparent PNG."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt_agg

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    vmin, vmax = value_range
    normalized = np.clip((artifact.array - vmin) / (vmax - vmin), 0.0, 1.0)
    rgba = matplotlib.colormaps[colormap](normalized)
    rgba[..., 3] = np.where(np.isnan(artifact.array), 0.0, 1.0)
    fig = plt_agg.figure(
        figsize=(artifact.grid_spec.width / 100, artifact.grid_spec.height / 100), dpi=100
    )
    fig.figimage(rgba, resize=False)
    fig.savefig(output_path, transparent=True)
    plt_agg.close(fig)
    return output_path

previews_dir = Path(output_directory) / "previews"
forest_preview = build_preview(
    forest_mask_path, settings["runtime"]["preview_max_dimension_pixels"], categorical=True
)
plt.figure(figsize=(6, 5))
plt.imshow(forest_preview.array, cmap="Greens", vmin=0, vmax=1)
plt.title("Forest mask (small preview)")
plt.axis("off")
plt.show()

hls_composite_paths = sorted(composites_dir.glob("hls_*.tif"))
hls_composite_paths = [p for p in hls_composite_paths if "_count" not in p.stem]
print(f"{len(hls_composite_paths)} yearly HLS picture(s) on disk")
if hls_composite_paths:
    hls_preview = build_preview(
        hls_composite_paths[-1], settings["runtime"]["preview_max_dimension_pixels"]
    )
    plt.figure(figsize=(6, 5))
    plt.imshow(hls_preview.array, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
    plt.title(f"{hls_composite_paths[-1].stem}: first index (small preview)")
    plt.axis("off")
    plt.show()


## 9. Calculate baseline and change

Build the multi-year baseline and compare each target year with that baseline.


In [ ]:
# calculate the multi-year baseline and change products
def compute_variable_baseline(
    name, yearly_layers, mad_scale, minimum_spread_floors, minimum_baseline_year_count
):
    """yearly_layers maps a before-year to that year's picture array for this one index."""
    years = sorted(yearly_layers)
    if len(years) < minimum_baseline_year_count:
        raise ValueError(
            f"only {len(years)} before-year picture(s) available for {name}, need at least {minimum_baseline_year_count}"
        )
    stack = np.stack([yearly_layers[year] for year in years], axis=0)
    median = np.nanmedian(stack, axis=0)
    # Use scaled MAD with a minimum floor to avoid unstable anomalies.
    mad = np.nanmedian(np.abs(stack - median), axis=0)
    floor = minimum_spread_floors[name]
    scaled_mad = mad_scale * mad
    robust_spread = np.where(scaled_mad < floor, floor, scaled_mad)
    return median, mad, robust_spread, years


def compute_difference(current, baseline):
    """Subtract the baseline from a target-year layer."""
    # Positive means the after-year is higher than the normal, before-year value.
    return current - baseline


def compute_percentage_change(
    difference, baseline, name, percentage_minimum_baseline_magnitude, percentage_excluded_variables
):
    """Left blank below a per-index minimum normal-value size; raises for the two indexes excluded entirely."""
    if name in percentage_excluded_variables:
        raise ValueError(f"{name} does not get a percentage-change result.")
    floor = percentage_minimum_baseline_magnitude[name]
    denominator = np.where(np.abs(baseline) > floor, np.abs(baseline), np.nan)
    return difference / denominator * 100.0


def compute_signed_robust_anomaly(difference, robust_spread):
    """Standardize change by the baseline robust spread."""
    # Preserve direction until disturbance magnitudes are created.
    return difference / robust_spread


def variable_receives_percentage(name, percentage_excluded_variables):
    """Check whether percentage change is enabled for a variable."""
    return name not in percentage_excluded_variables

def baseline_filenames(name):
    """Return output filenames for one variable's baseline rasters."""
    return {
        "baseline": f"baseline_{name}.tif",
        "mad": f"baseline_mad_{name}.tif",
        "spread": f"baseline_robust_spread_{name}.tif",
    }


def change_filenames(name, year):
    """Return output filenames for one variable and target year."""
    return {
        "difference": f"d_{name}_{year}.tif",
        "anomaly": f"z_{name}_{year}.tif",
        "percentage": f"pct_{name}_{year}.tif",
    }


def write_baseline_rasters(name, median, mad, robust_spread, grid_spec, output_directory):
    """Write baseline median, MAD, and robust-spread rasters."""
    output_directory = Path(output_directory)
    filenames = baseline_filenames(name)
    write_geotiff(output_directory / filenames["baseline"], median, grid_spec)
    write_geotiff(output_directory / filenames["mad"], mad, grid_spec)
    write_geotiff(output_directory / filenames["spread"], robust_spread, grid_spec)


def compute_change_products_windowed(
    name,
    year,
    baseline_path,
    robust_spread_path,
    target_composite_path,
    percentage_minimum_baseline_magnitude,
    percentage_excluded_variables,
    grid_spec,
    output_directory,
    block_pixels,
    target_band_index=1,
):
    """Write difference, percentage, and anomaly rasters one window at a time."""
    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)
    filenames = change_filenames(name, year)
    difference_path = output_directory / filenames["difference"]
    anomaly_path = output_directory / filenames["anomaly"]
    write_percentage = variable_receives_percentage(name, percentage_excluded_variables)
    percentage_path = output_directory / filenames["percentage"] if write_percentage else None

    profile = {
        "driver": "GTiff",
        "height": grid_spec.height,
        "width": grid_spec.width,
        "count": 1,
        "dtype": "float32",
        "crs": grid_spec.crs,
        "transform": grid_spec.transform,
        "nodata": np.nan,
    }
    windows = iter_grid_windows(grid_spec, block_pixels)
    with (
        rasterio.open(difference_path, "w", **profile) as difference_dst,
        rasterio.open(anomaly_path, "w", **profile) as anomaly_dst,
    ):
        percentage_dst = (
            rasterio.open(percentage_path, "w", **profile) if write_percentage else None
        )
        try:
            for window in windows:
                baseline_block = read_window(baseline_path, window, [1])[0]
                spread_block = read_window(robust_spread_path, window, [1])[0]
                target_block = read_window(target_composite_path, window, [target_band_index])[0]

                difference_block = compute_difference(target_block, baseline_block)
                anomaly_block = compute_signed_robust_anomaly(difference_block, spread_block)
                difference_dst.write(difference_block[np.newaxis, ...], window=window)
                anomaly_dst.write(anomaly_block[np.newaxis, ...], window=window)

                if percentage_dst is not None:
                    percentage_block = compute_percentage_change(
                        difference_block,
                        baseline_block,
                        name,
                        percentage_minimum_baseline_magnitude,
                        percentage_excluded_variables,
                    )
                    percentage_dst.write(percentage_block[np.newaxis, ...], window=window)
                del baseline_block, spread_block, target_block, difference_block, anomaly_block
        finally:
            if percentage_dst is not None:
                percentage_dst.close()
    return difference_path, anomaly_path, percentage_path


def _composite_path_for(name, year, output_directory, hls_indices):
    if name in hls_indices:
        return Path(output_directory) / "composites" / f"hls_{year}.tif"
    return Path(output_directory) / "composites" / f"opera_rtc_{year}.tif"


def _band_index_for(name, hls_indices, radar_indices):
    if name in hls_indices:
        return list(hls_indices).index(name) + 1
    return list(radar_indices).index(name) + 1


In [ ]:
# calculate optical baseline and change
baselines_dir = Path(output_directory) / "baselines"
change_dir = Path(output_directory) / "change"


# Create baseline and target-year change rasters for each variable.
def run_baseline_and_change_for_variables(variable_names):
    """Create baseline and change rasters for a list of variables."""
    for name in variable_names:
        band_index = _band_index_for(
            name, settings["hls"]["indices"], settings["opera_rtc"]["radar_indices"]
        )
        yearly_layers = {}
        for year in baseline_years:
            path = _composite_path_for(name, year, output_directory, settings["hls"]["indices"])
            if path.is_file():
                yearly_layers[year] = read_full_array(path, [band_index])[0]
        if not yearly_layers:
            print(f"{name}: no before-year picture found, skipped")
            continue
        median, mad, robust_spread, used_years = compute_variable_baseline(
            name,
            yearly_layers,
            settings["baseline"]["mad_scale"],
            settings["baseline"]["minimum_spread_floors"],
            settings["baseline"]["minimum_baseline_year_count"],
        )
        write_baseline_rasters(name, median, mad, robust_spread, grid_spec, baselines_dir)

        for year in target_years:
            target_path = _composite_path_for(
                name, year, output_directory, settings["hls"]["indices"]
            )
            if not target_path.is_file():
                continue
            baseline_filename_map = baseline_filenames(name)
            compute_change_products_windowed(
                name,
                year,
                baselines_dir / baseline_filename_map["baseline"],
                baselines_dir / baseline_filename_map["spread"],
                target_path,
                settings["change"]["percentage_minimum_baseline_magnitude"],
                settings["change"]["percentage_excluded_variables"],
                grid_spec,
                change_dir,
                settings["runtime"]["analysis_chunk_pixels"],
                target_band_index=band_index,
            )
        print(f"{name}: normal value from {used_years}, compared against {target_years}")


run_baseline_and_change_for_variables(list(settings["hls"]["indices"]))

optical_preview_variable = "ndvi"
optical_diff_path = (
    change_dir / change_filenames(optical_preview_variable, target_years[-1])["difference"]
)
if optical_diff_path.is_file():
    optical_diff_preview = build_preview(
        optical_diff_path, settings["runtime"]["preview_max_dimension_pixels"]
    )
    plt.figure(figsize=(6, 5))
    plt.imshow(optical_diff_preview.array, cmap="RdYlGn", vmin=-0.3, vmax=0.3)
    plt.title(f"{optical_diff_path.stem}: after minus before (small preview)")
    plt.axis("off")
    plt.show()


In [ ]:
# calculate radar baseline and change
# Calculate radar baseline and change.
run_baseline_and_change_for_variables(list(settings["opera_rtc"]["radar_indices"]))

radar_preview_variable = "rvi"
radar_diff_path = (
    change_dir / change_filenames(radar_preview_variable, target_years[-1])["difference"]
)
if radar_diff_path.is_file():
    radar_diff_preview = build_preview(
        radar_diff_path, settings["runtime"]["preview_max_dimension_pixels"]
    )
    plt.figure(figsize=(6, 5))
    plt.imshow(radar_diff_preview.array, cmap="RdBu_r", vmin=-0.3, vmax=0.3)
    plt.title(f"{radar_diff_path.stem}: after minus before (small preview)")
    plt.axis("off")
    plt.show()


## 10. Create and summarize disturbance scores

Combine optical and radar change, then create tables and figures.


In [ ]:
# combine optical and radar change into disturbance scores
def transform_magnitude(value, cap, normalize=True):
    """Keep positive disturbance, cap extreme values, and optionally scale to 0–1."""
    clipped = np.clip(value, 0, cap)
    if normalize:
        clipped = np.clip(clipped / float(cap), 0.0, 1.0)
    return clipped


def rms(values):
    """An equal-weight combined size across several numbers (root of the mean of the squares)."""
    total = sum(value**2 for value in values)
    return (total / len(values)) ** 0.5


def spectral_magnitude(name, anomaly, disturbance_direction, cap):
    """Convert a signed optical anomaly to disturbance magnitude."""
    # Only change in the configured disturbance direction contributes.
    direction = disturbance_direction[name]
    return transform_magnitude(direction * anomaly, cap)


def radar_magnitude(anomaly, cap):
    """Radar numbers count a change in either direction, not just one."""
    return transform_magnitude(np.abs(anomaly), cap)


def grouped_spectral_rms(anomalies, spectral_groups, disturbance_direction, cap):
    """Combines indices within each group first, then combines the groups."""
    domain_scores = {}
    for group_name, variable_names in spectral_groups.items():
        magnitudes = [
            spectral_magnitude(name, anomalies[name], disturbance_direction, cap)
            for name in variable_names
        ]
        domain_scores[group_name] = magnitudes[0] if len(magnitudes) == 1 else rms(magnitudes)
    combined = rms([domain_scores[name] for name in spectral_groups])
    return combined, domain_scores


def reduced_radar_rms(anomalies, radar_inputs, cap):
    """Combine the selected radar magnitudes with RMS."""
    magnitudes = [radar_magnitude(anomalies[name], cap) for name in radar_inputs]
    return rms(magnitudes)


def combined_rms(spectral, radar):
    """The two sensor types (optical and radar) are given equal weight in the final score."""
    return rms([spectral, radar])

candidates_dir = Path(output_directory) / "candidates"


# Combine optical and radar anomalies one raster window at a time.
def compute_composite_scores_windowed(
    year, anomaly_paths, composites_settings, grid_spec, output_directory, block_pixels
):
    """Write spectral, radar, and combined scores one window at a time."""
    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)
    spectral_path = output_directory / f"spectral_score_{year}.tif"
    radar_path = output_directory / f"radar_score_{year}.tif"
    combined_path = output_directory / f"combined_score_{year}.tif"
    profile = {
        "driver": "GTiff",
        "height": grid_spec.height,
        "width": grid_spec.width,
        "count": 1,
        "dtype": "float32",
        "crs": grid_spec.crs,
        "transform": grid_spec.transform,
        "nodata": np.nan,
    }
    spectral_variables = {
        name for names in composites_settings["spectral_groups"].values() for name in names
    }
    with (
        rasterio.open(spectral_path, "w", **profile) as spectral_dst,
        rasterio.open(radar_path, "w", **profile) as radar_dst,
        rasterio.open(combined_path, "w", **profile) as combined_dst,
    ):
        for window in iter_grid_windows(grid_spec, block_pixels):
            anomaly_blocks = {
                name: read_window(anomaly_paths[name], window, [1])[0]
                for name in spectral_variables | set(composites_settings["radar_inputs"])
            }
            spectral_score, _ = grouped_spectral_rms(
                anomaly_blocks,
                composites_settings["spectral_groups"],
                composites_settings["spectral_disturbance_direction"],
                composites_settings["z_cap"],
            )
            radar_score = reduced_radar_rms(
                anomaly_blocks, composites_settings["radar_inputs"], composites_settings["z_cap"]
            )
            combined_score = combined_rms(spectral_score, radar_score)
            spectral_dst.write(spectral_score[np.newaxis, ...].astype("float32"), window=window)
            radar_dst.write(radar_score[np.newaxis, ...].astype("float32"), window=window)
            combined_dst.write(combined_score[np.newaxis, ...].astype("float32"), window=window)
            del anomaly_blocks, spectral_score, radar_score, combined_score
    return spectral_path, radar_path, combined_path


for year in target_years:
    spectral_variables = {
        name for names in settings["composites"]["spectral_groups"].values() for name in names
    }
    needed_variables = spectral_variables | set(settings["composites"]["radar_inputs"])
    anomaly_paths = {
        name: change_dir / change_filenames(name, year)["anomaly"] for name in needed_variables
    }
    if not all(path.is_file() for path in anomaly_paths.values()):
        print(f"{year}: missing one or more comparison results, skipped")
        continue
    compute_composite_scores_windowed(
        year,
        anomaly_paths,
        settings["composites"],
        grid_spec,
        candidates_dir,
        settings["runtime"]["composite_block_pixels"],
    )
    print(f"{year}: wrote the disturbance score")


In [ ]:
# summarize the change rasters
def compute_raster_summary(
    raster_path,
    grid_spec,
    variable,
    year,
    block_pixels=512,
    histogram_bins=1024,
    value_range=(-1.0, 1.0),
):
    """Calculate raster count, mean, median, and spread without loading the full file."""
    count = 0
    total = 0.0
    total_sq = 0.0
    lo, hi = value_range
    histogram = np.zeros(histogram_bins, dtype="int64")
    for window in iter_grid_windows(grid_spec, block_pixels):
        block = read_window(raster_path, window, [1])[0]
        finite = block[np.isfinite(block)]
        if finite.size == 0:
            continue
        count += finite.size
        total += float(finite.astype("float64").sum())
        total_sq += float(np.square(finite, dtype="float64").sum())
        clipped = np.clip(finite, lo, hi)
        bin_indexes = np.clip(
            ((clipped - lo) / (hi - lo) * histogram_bins).astype("int64"), 0, histogram_bins - 1
        )
        histogram += np.bincount(bin_indexes, minlength=histogram_bins)

    area_km2 = count * (grid_spec.resolution_m**2) / 1e6
    bin_width = (hi - lo) / histogram_bins
    if count == 0:
        return {
            "variable": variable,
            "year": year,
            "mean": None,
            "median": None,
            "std": None,
            "valid_count": 0,
            "area_km2": 0.0,
        }
    mean = total / count
    variance = max(total_sq / count - mean**2, 0.0)
    std = math.sqrt(variance)
    cumulative = np.cumsum(histogram)
    median_bin = int(np.searchsorted(cumulative, count / 2))
    median = lo + (median_bin + 0.5) * bin_width
    return {
        "variable": variable,
        "year": year,
        "mean": mean,
        "median": median,
        "std": std,
        "valid_count": count,
        "area_km2": area_km2,
    }


def bounded_sample_for_distribution(
    raster_path, grid_spec, max_sample_size, random_seed, block_pixels=512
):
    """A repeatable, windowed random sample of real pixel values, never every pixel in the picture."""
    rng = np.random.default_rng(random_seed)
    reservoir = np.empty(max_sample_size, dtype="float64")
    seen = 0
    filled = 0
    for window in iter_grid_windows(grid_spec, block_pixels):
        block = read_window(raster_path, window, [1])[0]
        finite = block[np.isfinite(block)].astype("float64")
        for value in finite:
            if filled < max_sample_size:
                reservoir[filled] = value
                filled += 1
            else:
                j = rng.integers(0, seen + 1)
                if j < max_sample_size:
                    reservoir[j] = value
            seen += 1
    return reservoir[:filled]

chart_statistics_rows = []
for variable_name in list(settings["hls"]["indices"]) + list(
    settings["opera_rtc"]["radar_indices"]
):
    for year in target_years:
        anomaly_path = change_dir / change_filenames(variable_name, year)["anomaly"]
        if anomaly_path.is_file():
            chart_statistics_rows.append(
                compute_raster_summary(
                    anomaly_path,
                    grid_spec,
                    variable_name,
                    year,
                    block_pixels=settings["runtime"]["analysis_chunk_pixels"],
                )
            )

if chart_statistics_rows:
    chart_statistics_table = pd.DataFrame(chart_statistics_rows)
    display(
        chart_statistics_table[
            ["variable", "year", "mean", "median", "std", "valid_count", "area_km2"]
        ]
    )
else:
    chart_statistics_table = pd.DataFrame(
        columns=["variable", "year", "mean", "median", "std", "valid_count"]
    )
    print("no comparison results yet, run Step 9 first")


In [ ]:
# plot yearly change and index distributions
figures_dir = Path(output_directory) / "figures"
for variable_name in ("ndvi", "nbr"):
    subset = chart_statistics_table[chart_statistics_table["variable"] == variable_name].dropna(
        subset=["year"]
    )
    subset = subset.sort_values("year")
    if subset.empty:
        continue
    baseline_subset = subset[subset["year"].isin(baseline_years)]
    target_subset = subset[subset["year"].isin(target_years)]

    fig, ax = plt.subplots(figsize=(6, 4))
    if not baseline_subset.empty:
        baseline_median = baseline_subset["median"].median()
        baseline_spread = baseline_subset["median"].std() if len(baseline_subset) > 1 else 0.0
        ax.axhspan(
            baseline_median - baseline_spread,
            baseline_median + baseline_spread,
            color="tab:blue",
            alpha=0.15,
            label="Before-years typical range",
        )
        ax.plot(
            baseline_subset["year"],
            baseline_subset["median"],
            "o-",
            color="tab:blue",
            label="Before years",
        )
    if not target_subset.empty:
        ax.plot(
            target_subset["year"],
            target_subset["median"],
            "o-",
            color="tab:red",
            label="After years",
        )
    ax.set_xlabel("Year")
    ax.set_ylabel(f"{variable_name} (middle value)")
    ax.set_title(f"Yearly change over time: {variable_name}")
    ax.legend()
    fig.tight_layout()
    figure_path = figures_dir / f"temporal_summary_{variable_name}.png"
    figure_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_path)
    plt.close(fig)
    display(Image(filename=str(figure_path)))

distribution_sample_size = settings["plots"]["distribution_sample_size"]
distribution_seed = settings["plots"]["distribution_seed"]
distribution_samples = {}
for variable_name in ("ndvi", "nbr"):
    for year in target_years:
        anomaly_path = change_dir / change_filenames(variable_name, year)["anomaly"]
        if anomaly_path.is_file():
            distribution_samples[f"z_{variable_name}_{year}"] = bounded_sample_for_distribution(
                anomaly_path,
                grid_spec,
                distribution_sample_size,
                distribution_seed,
                block_pixels=settings["runtime"]["analysis_chunk_pixels"],
            )

if distribution_samples:
    fig, ax = plt.subplots(figsize=(max(4, len(distribution_samples) * 1.2), 4))
    labels = list(distribution_samples.keys())
    ax.boxplot([distribution_samples[label] for label in labels], tick_labels=labels)
    ax.set_title(
        f"How values are spread out (sample of up to {distribution_sample_size} pixels each)"
    )
    fig.tight_layout()
    boxplot_path = figures_dir / "distribution_z_ndvi_nbr.png"
    fig.savefig(boxplot_path)
    plt.close(fig)
    display(Image(filename=str(boxplot_path)))


## 11. Run optional Local Moran's I

Aggregate the selected score to the chosen support resolution and identify significant spatial clusters.


In [ ]:
# Define the optional Local Moran helpers. This long helper cell is collapsed by default.
CLUSTER_LABELS = {
    0: "not_significant",
    1: "high_high",
    2: "low_high",
    3: "low_low",
    4: "high_low",
    5: "excluded_no_neighbors",
}


def resolve_lisa_raster(variable, year):
    names = {
        "index_composite_combined": f"combined_score_{year}.tif",
        "index_composite_spectral": f"spectral_score_{year}.tif",
        "index_composite_radar": f"radar_score_{year}.tif",
    }
    return Path(output_directory) / "candidates" / names[variable]


def aggregate_lisa_grid(score_path):
    """Average score pixels into the configured Local Moran support grid."""
    support_m = settings["lisa"]["support_resolution_m"]
    min_valid = settings["lisa"]["minimum_valid_fraction"]
    factor = support_m / grid_spec.resolution_m
    if not np.isclose(factor, round(factor)) or factor < 1:
        raise ValueError("Local Moran support must be a whole-number multiple of pixel size.")
    factor = int(round(factor))

    height = math.ceil(grid_spec.height / factor)
    width = math.ceil(grid_spec.width / factor)
    values = np.full((height, width), np.nan, dtype="float32")
    valid = np.zeros((height, width), dtype=bool)
    block_pixels = max(
        factor,
        (settings["runtime"]["composite_block_pixels"] // factor) * factor,
    )

    for window in iter_grid_windows(grid_spec, block_pixels):
        block = read_window(score_path, window, [1])[0].astype("float32")
        pad_rows = (-block.shape[0]) % factor
        pad_cols = (-block.shape[1]) % factor
        if pad_rows or pad_cols:
            block = np.pad(
                block,
                ((0, pad_rows), (0, pad_cols)),
                constant_values=np.nan,
            )
        grouped = block.reshape(
            block.shape[0] // factor,
            factor,
            block.shape[1] // factor,
            factor,
        )
        finite = np.isfinite(grouped)
        counts = finite.sum(axis=(1, 3))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            means = np.nanmean(grouped, axis=(1, 3)).astype("float32")
        keep = counts / (factor * factor) >= min_valid
        means[~keep] = np.nan

        row = int(window.row_off) // factor
        col = int(window.col_off) // factor
        values[row : row + means.shape[0], col : col + means.shape[1]] = means
        valid[row : row + means.shape[0], col : col + means.shape[1]] = keep

    support_transform = grid_spec.transform * Affine.scale(factor, factor)
    return values, valid, support_transform, support_m


def run_local_moran_for_year(score_path, year):
    """Calculate Local Moran's I and write the significant cluster raster."""
    import esda
    import libpysal
    from scipy.stats import false_discovery_control

    values, valid, support_transform, support_m = aggregate_lisa_grid(score_path)
    rows, cols = np.where(valid)
    if len(rows) > settings["lisa"]["maximum_units"]:
        raise RuntimeError(
            f"Local Moran has {len(rows):,} units. Increase lisa_support_resolution_m."
        )
    if len(rows) == 0:
        raise RuntimeError("No valid cells are available for Local Moran's I.")

    # Build queen or rook neighbors from the regular support grid.
    lookup = np.full(values.shape, -1, dtype="int64")
    lookup[rows, cols] = np.arange(len(rows))
    offsets = (
        [(-1, 0), (0, -1), (0, 1), (1, 0)]
        if settings["lisa"]["weights_method"] == "rook"
        else [
            (-1, -1), (-1, 0), (-1, 1),
            (0, -1),             (0, 1),
            (1, -1),  (1, 0),   (1, 1),
        ]
    )
    neighbors = {}
    for unit, (row, col) in enumerate(zip(rows, cols, strict=True)):
        neighbors[unit] = [
            int(lookup[row + drow, col + dcol])
            for drow, dcol in offsets
            if (
                0 <= row + drow < values.shape[0]
                and 0 <= col + dcol < values.shape[1]
                and lookup[row + drow, col + dcol] >= 0
            )
        ]

    islands = {unit for unit, nearby in neighbors.items() if not nearby}
    keep = [unit for unit in range(len(rows)) if unit not in islands]
    if not keep:
        raise RuntimeError("Every Local Moran unit has no neighbors.")

    remap = {old: new for new, old in enumerate(keep)}
    fitted_neighbors = {
        remap[old]: [remap[n] for n in neighbors[old] if n in remap]
        for old in keep
    }
    weights = libpysal.weights.W(fitted_neighbors, silence_warnings=True)
    fitted_values = values[rows[keep], cols[keep]].astype("float64")

    fitted = esda.moran.Moran_Local(
        fitted_values,
        weights,
        transformation="r" if settings["lisa"]["row_standardize"] else "b",
        permutations=lisa_permutations,
        n_jobs=settings["lisa"]["n_jobs"],
        keep_simulations=False,
        seed=settings["lisa"]["random_seed"],
        alternative=settings["lisa"]["alternative"],
    )
    adjusted_p = false_discovery_control(np.asarray(fitted.p_sim), method="bh")
    significant = adjusted_p < settings["lisa"]["alpha"]

    # esda quadrants: 1=HH, 2=LH, 3=LL, 4=HL.
    cluster_codes = np.where(significant, np.asarray(fitted.q), 0).astype("int16")
    cluster_grid = np.full(values.shape, -1, dtype="int16")
    if islands:
        island_list = sorted(islands)
        cluster_grid[rows[island_list], cols[island_list]] = 5
    cluster_grid[rows[keep], cols[keep]] = cluster_codes

    lisa_dir.mkdir(parents=True, exist_ok=True)
    cluster_path = lisa_dir / f"lisa_cluster_{lisa_variable}_{year}.tif"
    with rasterio.open(
        cluster_path,
        "w",
        driver="GTiff",
        height=cluster_grid.shape[0],
        width=cluster_grid.shape[1],
        count=1,
        dtype="int16",
        crs=grid_spec.crs,
        transform=support_transform,
        nodata=-1,
    ) as destination:
        destination.write(cluster_grid, 1)

    cell_area_km2 = (support_m ** 2) / 1_000_000
    counts = [
        {
            "year": year,
            "cluster": label,
            "unit_count": int(np.sum(cluster_grid == code)),
            "area_km2": int(np.sum(cluster_grid == code)) * cell_area_km2,
        }
        for code, label in CLUSTER_LABELS.items()
    ]
    pd.DataFrame(counts).to_csv(
        tables_dir / f"lisa_cluster_counts_{lisa_variable}_{year}.csv",
        index=False,
    )

    return {
        "year": year,
        "units": len(keep),
        "islands": len(islands),
        "significant_units": int(significant.sum()),
        "support_resolution_m": support_m,
        "permutations": lisa_permutations,
        "cluster_path": str(cluster_path),
    }


In [ ]:
# run Local Moran's I only when requested
lisa_dir = Path(output_directory) / "lisa"
lisa_variable = settings["lisa"]["variable"]
lisa_results = []

if run_local_moran:
    for year in target_years:
        score_path = resolve_lisa_raster(lisa_variable, year)
        if not score_path.is_file():
            print(f"{year}: score raster not found, Local Moran skipped")
            continue
        result = run_local_moran_for_year(score_path, year)
        lisa_results.append(result)
        print(
            f"{year}: {result['significant_units']:,} significant units "
            f"from {result['units']:,} analyzed"
        )
else:
    print("Local Moran's I skipped")

lisa_summary_df = pd.DataFrame(lisa_results)
if not lisa_summary_df.empty:
    lisa_summary_df.to_csv(tables_dir / f"lisa_summary_{lisa_variable}.csv", index=False)
    display(lisa_summary_df)


## 12. Display and export the results

Show the latest disturbance score and cluster map, then save the run configuration, software information, and output inventory.


In [ ]:
# display the disturbance score and Local Moran cluster map
def raster_overlay_layer(preview_artifact, output_png_path, colormap, value_range, label):
    """Create the files and bounds needed for a map overlay."""
    png_path = preview_to_png(preview_artifact, output_png_path, colormap, value_range)
    bounds = preview_bounds_4326(preview_artifact)
    return {"png_path": str(png_path), "bounds_4326": bounds, "label": label}


def _png_data_url(png_path):
    encoded = base64.b64encode(Path(png_path).read_bytes()).decode()
    return f"data:image/png;base64,{encoded}"


result_map = Map(center=map_center, zoom=10, basemap=basemaps.Esri.WorldImagery)

score_paths = sorted(candidates_dir.glob("combined_score_*.tif"))
if score_paths:
    score_path = score_paths[-1]
    score_preview = build_preview(score_path, settings["runtime"]["preview_max_dimension_pixels"])
    score_layer = raster_overlay_layer(
        score_preview,
        figures_dir / f"{score_path.stem}.png",
        "inferno",
        (0.0, 1.0),
        "Disturbance (0-1, not a probability, not confirmed debris)",
    )
    result_map.add(
        ImageOverlay(
            url=_png_data_url(score_layer["png_path"]),
            bounds=score_layer["bounds_4326"],
            name=score_layer["label"],
        )
    )

lisa_cluster_paths = sorted(lisa_dir.glob(f"lisa_cluster_{lisa_variable}_*.tif"))
if lisa_cluster_paths:
    cluster_path = lisa_cluster_paths[-1]
    cluster_preview = build_preview(
        cluster_path, settings["runtime"]["preview_max_dimension_pixels"], categorical=True
    )
    # Display the multiple-testing-adjusted cluster map.
    cluster_layer = raster_overlay_layer(
        cluster_preview,
        figures_dir / f"{cluster_path.stem}.png",
        "Set1",
        (0, 5),
        f"Neighbor clusters ({cluster_path.stem}), adjusted for many comparisons",
    )
    result_map.add(
        ImageOverlay(
            url=_png_data_url(cluster_layer["png_path"]),
            bounds=cluster_layer["bounds_4326"],
            name=cluster_layer["label"],
        )
    )

result_map.add(LayersControl())
display(result_map)


In [ ]:
# write the output inventory, software versions, and run summary
def write_output_inventory_csv(output_directory):
    """Write a table describing every generated output file."""
    output_directory = Path(output_directory)
    inventory_path = output_directory / "tables" / "output_inventory.csv"
    rows = []
    for path in sorted(output_directory.rglob("*")):
        if not path.is_file() or path == inventory_path:
            continue
        relative_path = path.relative_to(output_directory)
        stat = path.stat()
        rows.append(
            {
                "relative_path": str(relative_path),
                "category": relative_path.parts[0] if relative_path.parts else "",
                "size_bytes": stat.st_size,
                "modified_at": datetime.fromtimestamp(stat.st_mtime, tz=UTC).isoformat(),
            }
        )
    inventory_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows, columns=["relative_path", "category", "size_bytes", "modified_at"]).to_csv(
        inventory_path, index=False
    )
    return inventory_path


def collect_software_versions():
    """Collect the operating system, hardware summary, Python, and package versions."""
    import os
    import platform
    from importlib.metadata import PackageNotFoundError, version

    versions = {
        "python": sys.version.split()[0],
        "operating_system": platform.platform(),
        "machine": platform.machine(),
        "processor": platform.processor() or "not reported",
        "logical_cpu_count": os.cpu_count(),
    }
    try:
        import psutil

        versions["physical_cpu_count"] = psutil.cpu_count(logical=False)
        versions["memory_gb"] = round(psutil.virtual_memory().total / (1024**3), 2)
    except Exception:
        versions["physical_cpu_count"] = "not available"
        versions["memory_gb"] = "not available"

    for package_name in (
        "earthaccess",
        "geopandas",
        "rasterio",
        "shapely",
        "pyproj",
        "numpy",
        "pandas",
        "matplotlib",
        "ipyleaflet",
        "scipy",
        "libpysal",
        "esda",
        "pyyaml",
        "psutil",
    ):
        try:
            versions[package_name] = version(package_name)
        except PackageNotFoundError:
            versions[package_name] = "not installed"
    return versions


def build_run_manifest(
    settings,
    aoi_method,
    aoi_source_description,
    aoi_bounds_4326,
    area_km2,
    grid_spec,
    software_versions,
):
    """Build a summary dictionary for the completed run."""
    return {
        "schema_version": 1,
        "generated_at": datetime.now(UTC).isoformat(),
        "aoi_method": aoi_method,
        "aoi_source": aoi_source_description,
        "aoi_bounds_4326": list(aoi_bounds_4326),
        "aoi_area_km2": area_km2,
        "baseline_years": settings["time"]["baseline_years"],
        "target_years": settings["time"]["target_years"],
        "season_start": settings["time"]["season_start"],
        "season_end": settings["time"]["season_end"],
        "maximum_cloud_cover_percent": settings["hls"]["maximum_granule_cloud_cover_percent"],
        "grid": {
            "crs": grid_spec.crs,
            "resolution_m": grid_spec.resolution_m,
            "width": grid_spec.width,
            "height": grid_spec.height,
        },
        "software_versions": software_versions,
        "settings": settings,
    }


In [ ]:
# save reproducibility files and print the final run summary
software_versions = collect_software_versions()
pd.DataFrame(list(software_versions.items()), columns=["package", "version"]).to_csv(
    config_dir / "software_versions.csv", index=False
)

run_manifest = build_run_manifest(
    settings,
    aoi_method,
    aoi_source_description,
    tuple(dissolved_aoi_4326.total_bounds),
    area_km2,
    grid_spec,
    software_versions,
)
run_manifest_path = config_dir / "run_manifest.json"
run_manifest_path.write_text(json.dumps(run_manifest, indent=2, default=str))

output_inventory_path = write_output_inventory_csv(output_directory)
print(f"wrote {output_inventory_path}")
if output_inventory_path.is_file():
    display(pd.read_csv(output_inventory_path).head(20))

print(f"area: {run_manifest['aoi_method']} ({run_manifest['aoi_area_km2']:.1f} square km)")
print(
    f"before years: {run_manifest['baseline_years']}, after years: {run_manifest['target_years']}"
)
print(
    f"calculation grid: {run_manifest['grid']['width']} x {run_manifest['grid']['height']} pixels at {run_manifest['grid']['resolution_m']} m"
)
print(f"settings and software versions saved under: {config_dir}")
